In [1]:
!pip install -q transformers datasets accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU available: True
Device: Tesla T4


In [3]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
import os

# Simple preprocess: tokenize input and target
def preprocess_batch(batch, tokenizer, max_input=256, max_target=64):
    inputs = tokenizer(
        batch["pattern"] + " <SEP> " + batch["buggy"],
        truncation=True,
        padding="max_length",
        max_length=max_input,
    )
    targets = tokenizer(
        batch["fixed"],
        truncation=True,
        padding="max_length",
        max_length=max_target,
    )
    batch["input_ids"] = inputs.input_ids
    batch["attention_mask"] = inputs.attention_mask
    batch["labels"] = targets.input_ids
    return batch

In [4]:
# ─── Step 4: Load a sample of TSSB-3M-ext ──────────────────────────────────────

from datasets import load_dataset

# 1) Load a 1% sample for quick prototyping
ds = load_dataset("zirui3/TSSB-3M-ext", split="train[:1%]")

# 2) Inspect available columns
print("Examples loaded:", len(ds))
print("Available columns:", ds.column_names)

# 3) Rename to the names our pipeline expects
ds = ds.rename_column("before",         "buggy")   # buggy snippet
ds = ds.rename_column("after",          "fixed")   # fixed snippet
ds = ds.rename_column("sstub_pattern",  "pattern") # the SStuB pattern label

# 4) Select only the three fields we need
ds = ds.select_columns(["pattern", "buggy", "fixed"])

# 5) Shuffle for randomness
ds = ds.shuffle(seed=42)

# Final sanity check
print(ds.column_names)
print(ds[0])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/2.62k [00:00<?, ?B/s]

data.jsonl.gz:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3341232 [00:00<?, ? examples/s]

Examples loaded: 33412
Available columns: ['project', 'commit_sha', 'parent_sha', 'file_path', 'project_url', 'likely_bug', 'comodified', 'in_function', 'diff', 'before', 'after', 'sstub_pattern', 'edit_script', 'key', 'commit_message', 'files']
['pattern', 'buggy', 'fixed']
{'pattern': 'CHANGE_STRING_LITERAL', 'buggy': 'msg = "No data found for {ecosystem} Package " "{package}/{version}" . format ( ecosystem = ecosystem , package = package , version = version )', 'fixed': 'msg = "No data found for {ecosystem} package " "{package}/{version}" . format ( ecosystem = ecosystem , package = package , version = version )'}


In [5]:
# ─── Step 5: Tokenizer & Model Setup (fixed for batched inputs) ────────────────

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
import torch

# 1) Load CodeT5
model_name = "Salesforce/codet5-base"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 2) Batched preprocess function
def preprocess_batch(batch, tokenizer, max_input=256, max_target=64):
    # build list of "<pattern> <SEP> <buggy>"
    inputs = tokenizer(
        [p + " <SEP> " + b for p, b in zip(batch["pattern"], batch["buggy"])],
        truncation=True,
        padding="max_length",
        max_length=max_input,
    )
    # tokenize targets
    targets = tokenizer(
        batch["fixed"],
        truncation=True,
        padding="max_length",
        max_length=max_target,
    )
    batch["input_ids"]      = inputs["input_ids"]
    batch["attention_mask"] = inputs["attention_mask"]
    batch["labels"]         = targets["input_ids"]
    return batch

# 3) Apply to the dataset
ds = ds.map(lambda batch: preprocess_batch(batch, tokenizer), batched=True)

# 4) Create collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 5) Sanity check
print("One tokenized example:", {k: v[:5] for k, v in ds[0].items() if k in ["input_ids","labels"]})
print("Ready on device:", "GPU" if torch.cuda.is_available() else "CPU")

tokenizer_config.json:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/703k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/294k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/12.5k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Map:   0%|          | 0/33412 [00:00<?, ? examples/s]

One tokenized example: {'input_ids': [1, 14473, 67, 5804, 67], 'labels': [1, 3576, 273, 315, 2279]}
Ready on device: GPU


In [6]:
# ─── Step 6 (Revised): TrainingArguments, Trainer + Accuracy ────────────────

import os, torch
from transformers import Trainer, TrainingArguments
from tqdm import tqdm
from transformers import pipeline

# 1) Create output dir
output_dir = "codet5-error-hints"
os.makedirs(output_dir, exist_ok=True)

# 2) TrainingArguments (4 epochs)
training_args = TrainingArguments(
    output_dir=output_dir,
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_steps=500,
    save_steps=1000,
    logging_steps=200,
    num_train_epochs=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    save_total_limit=2,
    fp16=torch.cuda.is_available()
)

# 3) Split dataset 90/10
train_size = int(0.9 * len(ds))
train_ds = ds.select(range(train_size))
eval_ds  = ds.select(range(train_size, len(ds)))

# 4) Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=data_collator
)

# 5) Train
train_output = trainer.train()

# 6) Save model & tokenizer
trainer.save_model("codet5-python-error-hints")
tokenizer.save_pretrained("codet5-python-error-hints")

# 7) Reload a generation pipeline for inference
pipe = pipeline(
    "text2text-generation",
    model="codet5-python-error-hints",
    tokenizer="codet5-python-error-hints",
    device=0
)

# 8) Manual Exact-Match accuracy on the eval split
match_count = 0
for ex in tqdm(eval_ds, desc="Computing accuracy"):
    prompt = ex["pattern"] + " <SEP> " + ex["buggy"]
    pred   = pipe(prompt, max_length=64)[0]["generated_text"].strip()
    if pred == ex["fixed"].strip():
        match_count += 1

accuracy = match_count / len(eval_ds) * 100
print(f"\n➡️ Exact-match accuracy on eval set: {accuracy:.2f}%")

<ipython-input-6-a83eff717022>:35: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: patelkaran0602 (patelkaran0602-george-washington-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
200,0.405200
400,0.223300
600,0.224100
800,0.217100
1000,0.217100
1200,0.214700
1400,0.206300
1600,0.201100
1800,0.206200
2000,0.197800


Step,Training Loss
200,0.405200
400,0.223300
600,0.224100
800,0.217100
1000,0.217100
1200,0.214700
1400,0.206300
1600,0.201100
1800,0.206200
2000,0.197800


Device set to use cuda:0
Computing accuracy: 100%|██████████| 3342/3342 [29:47<00:00,  1.87it/s]


➡️ Exact-match accuracy on eval set: 23.07%


In [7]:
# 1) Save via Trainer
SAVE_DIR = "codet5-python-error-hints"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"✅ Saved model and tokenizer locally to '{SAVE_DIR}/'")

# 2) Zip for download
!zip -q -r codet5_finetuned.zip {SAVE_DIR}
print("✅ Zipped model to codet5_finetuned.zip")

# 3) (Optional) Download the zip to your machine
from google.colab import files
files.download("codet5_finetuned.zip")

# 4) (Optional) Persist in Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DRIVE_DIR = "/content/drive/MyDrive/codet5_python_model"
model.save_pretrained(DRIVE_DIR)
tokenizer.save_pretrained(DRIVE_DIR)
print(f"✅ Also saved model and tokenizer to Google Drive at '{DRIVE_DIR}/'")

✅ Saved model and tokenizer locally to 'codet5-python-error-hints/'
✅ Zipped model to codet5_finetuned.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Mounted at /content/drive
✅ Also saved model and tokenizer to Google Drive at '/content/drive/MyDrive/codet5_python_model/'


In [8]:
# Install evaluation libraries
!pip install -q evaluate sacrebleu rouge-score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 8.7 MB/s eta 0:00:00


In [10]:
# ─── Step 7: Batched Evaluation Metrics ────────────────────────────────────────

# 1) Install evaluation backends
!pip install -q evaluate sacrebleu rouge-score

import evaluate
import numpy as np
from transformers import pipeline

# 2) Build prompts & references from your eval split
prompts = [p + " <SEP> " + b for p,b in zip(eval_ds["pattern"], eval_ds["buggy"])]
refs    = [f.strip() for f in eval_ds["fixed"]]

# 3) Reload your saved model as a batched pipeline
pipe = pipeline(
    "text2text-generation",
    model="codet5-python-error-hints",
    tokenizer="codet5-python-error-hints",
    device=0,        # or -1 for CPU
    batch_size=16,   # runs 16 examples at once
    truncation=True,
    max_length=64
)

# 4) Generate all predictions in one go
outs = pipe(prompts)
preds = [o["generated_text"].strip() for o in outs]

# 5) Compute Exact-Match “accuracy”
exact_match = np.mean([p==r for p,r in zip(preds, refs)]) * 100

# 6) Load BLEU & ROUGE
bleu  = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

bleu_res  = bleu.compute(predictions=preds, references=[[r] for r in refs])
rouge_res = rouge.compute(predictions=preds, references=refs, use_stemmer=True)

# 7) Print everything
print(f"➡️ Exact-Match Accuracy:  {exact_match:.2f}%")
print(f"➡️ BLEU-4 Score:         {bleu_res['score']:.2f}")
print(f"➡️ ROUGE-1 F1 Score:     {rouge_res['rouge1']*100:.2f}%")
print(f"➡️ ROUGE-L F1 Score:     {rouge_res['rougeL']*100:.2f}%")

Device set to use cuda:0


➡️ Exact-Match Accuracy:  23.07%
➡️ BLEU-4 Score:         79.96
➡️ ROUGE-1 F1 Score:     85.37%
➡️ ROUGE-L F1 Score:     84.80%


In [19]:
!pip install -q openai datasets sentencepiece tqdm

In [22]:
import torch
print("GPU:", torch.cuda.is_available())

GPU: True


In [ ]:
import openai
from transformers import pipeline

openai.api_key = "API Key"   

pipe = pipeline(
    "text2text-generation",
    model="codet5-python-error-hints",
    tokenizer="codet5-python-error-hints",
    device=0,
    truncation=True,
    max_length=64
)

def explain_fix_as_hint(buggy, fix):
    resp = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role":"system","content":"You are a patient Python tutor."},
            {"role":"user","content":(
                f"The student wrote this code and got an error:\n```python\n{buggy}\n```\n"
                f"Our model’s suggested fix is:\n```python\n{fix}\n```\n"
                "Provide a concise hint (not full solution)."
            )}
        ],
        temperature=0.7,
        max_tokens=60
    )
    return resp.choices[0].message.content.strip()


Device set to use cuda:0


In [24]:
# --- Your buggy code and IDE error ---
student_code = """
# Paste your buggy code here—for example:
def add(a, b)
    return a + b
"""
error_msg = "SyntaxError: invalid syntax on line 2"

# --- Generate fix and hint ---
prompt = student_code + " <SEP> " + error_msg
fix  = pipe(prompt)[0]["generated_text"].strip()
hint = explain_fix_as_hint(student_code, fix)

# --- Show results ---
print("🔧 Buggy code:\n", student_code)
print("❌ Error message:\n", error_msg)
print("📋 Model-generated fix:\n", fix)
print("💡 Mentor-style hint:\n", hint)

🔧 Buggy code:
 
# Paste your buggy code here—for example:
def add(a, b)
    return a + b

❌ Error message:
 SyntaxError: invalid syntax on line 2
📋 Model-generated fix:
 Paste your buggy code here—for example:
def add(a, b, c)
    return a + b
💡 Mentor-style hint:
 The error is likely due to the missing colon `:` at the end of the function definition.
